# 1. Import Libraries

In [1]:
import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# 2. Define Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATA_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "data"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"

DATA_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_SCORED_PATH = PROCESSED_DIR / "train_scored.csv"
VALID_SCORED_PATH = PROCESSED_DIR / "valid_scored.csv"
TEST_SCORED_PATH = PROCESSED_DIR / "test_scored.csv"

print("Project root:", PROJECT_ROOT)
print("Train scored exists:", TRAIN_SCORED_PATH.exists())
print("Valid scored exists:", VALID_SCORED_PATH.exists())
print("Test scored exists :", TEST_SCORED_PATH.exists())

Project root: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator
Train scored exists: True
Valid scored exists: True
Test scored exists : True


# 3. Load Data

In [4]:
train_scored = pd.read_csv(TRAIN_SCORED_PATH, low_memory=False, parse_dates=["issue_date"])
valid_scored = pd.read_csv(VALID_SCORED_PATH, low_memory=False, parse_dates=["issue_date"])
test_scored = pd.read_csv(TEST_SCORED_PATH, low_memory=False, parse_dates=["issue_date"])

print("Train scored:", train_scored.shape)
print("Valid scored:", valid_scored.shape)
print("Test scored :", test_scored.shape)

train_scored.head()

Train scored: (277221, 32)
Valid scored: (59404, 32)
Test scored : (59405, 32)


,default_flag,loan_amnt,term_months,annual_inc_log,dti_capped,emp_length_num,emp_length_missing,open_acc,pub_rec,revol_bal_log,revol_util_capped,revol_util_missing,total_acc,mort_acc,mort_acc_missing,pub_rec_bankruptcies,pub_rec_bankruptcies_missing,credit_history_length_capped,home_ownership,verification_status,purpose,initial_list_status,application_type,loan_status,grade,sub_grade,int_rate,installment,issue_date,pd_raw,pd_sigmoid,calibrated_pd
0,0,"15,000.0000",36.0000,11.3145,9.9200,10.0000,0,21.0000,0.0000,9.6262,37.5000,0,36.0000,0.0000,0,0.0000,0,26.8337,RENT,Not Verified,debt_consolidation,f,INDIVIDUAL,Fully Paid,A,A5,8.9000,476.3000,2013-03-01,0.1010,0.1094,0.1010
1,0,"15,000.0000",36.0000,10.8590,13.7100,2.0000,0,5.0000,0.0000,8.6985,54.5000,0,13.0000,0.0000,0,0.0000,0,12.5804,RENT,Not Verified,debt_consolidation,w,INDIVIDUAL,Fully Paid,A,A4,7.2600,464.9500,2015-05-01,0.1436,0.1366,0.1436
2,0,"19,800.0000",36.0000,12.2784,9.8300,9.0000,0,11.0000,0.0000,10.0388,82.7000,0,30.0000,5.0000,0,0.0000,0,18.6667,MORTGAGE,Not Verified,debt_consolidation,w,INDIVIDUAL,Fully Paid,B,B1,9.6700,635.8300,2014-04-01,0.0820,0.0988,0.0820
3,0,"35,000.0000",36.0000,11.3386,18.6300,1.0000,0,6.0000,0.0000,9.3424,49.0000,0,22.0000,2.0000,0,0.0000,0,13.9986,OWN,Source Verified,debt_consolidation,w,INDIVIDUAL,Fully Paid,C,C1,12.2900,"1,167.3600",2015-10-01,0.1813,0.1654,0.1813
4,0,"28,000.0000",60.0000,11.3504,26.6100,10.0000,0,14.0000,0.0000,10.3263,72.2000,0,29.0000,2.0000,0,0.0000,0,14.5818,MORTGAGE,Verified,debt_consolidation,f,INDIVIDUAL,Fully Paid,E,E1,18.9900,726.1900,2014-05-01,0.3559,0.3594,0.3559


In [5]:
# Confirm PD column and targe

required_cols = ["default_flag", "calibrated_pd"]

for df_name, df in {
    "train_scored": train_scored,
    "valid_scored": valid_scored,
    "test_scored": test_scored
}.items():
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{df_name} missing columns: {missing}")

for name, data in {
    "train": train_scored,
    "valid": valid_scored,
    "test": test_scored
}.items():
    print("=" * 80)
    print(name.upper())
    print("Actual default rate:", data["default_flag"].mean())
    print("Average calibrated PD:", data["calibrated_pd"].mean())
    print("PD min:", data["calibrated_pd"].min())
    print("PD max:", data["calibrated_pd"].max())

TRAIN
Actual default rate: 0.19612872040718415
Average calibrated PD: 0.1961686455959433
PD min: 0.016728075
PD max: 0.82553184
VALID
Actual default rate: 0.1961315736314053
Average calibrated PD: 0.19591612916993803
PD min: 0.021245528
PD max: 0.7473527
TEST
Actual default rate: 0.19612827203097383
Average calibrated PD: 0.19604216649647332
PD min: 0.017098079
PD max: 0.7678783


In [7]:
# Baseline PD decile diagnostic

def pd_decile_summary(df, pd_col="calibrated_pd", target_col="default_flag", n_bins=10):
    temp = df[[pd_col, target_col]].copy()
    
    # Highest risk = decile 1
    temp["risk_decile"] = pd.qcut(
        temp[pd_col].rank(method="first", ascending=False),
        q=n_bins,
        labels=[f"D{i}" for i in range(1, n_bins + 1)]
    )
    
    summary = (
        temp
        .groupby("risk_decile", observed=True)
        .agg(
            n_loans=(target_col, "size"),
            avg_pd=(pd_col, "mean"),
            actual_default_rate=(target_col, "mean"),
            defaults=(target_col, "sum")
        )
        .reset_index()
    )
    
    summary["lift"] = summary["actual_default_rate"] / temp[target_col].mean()
    return summary


valid_decile_summary = pd_decile_summary(valid_scored)
test_decile_summary = pd_decile_summary(test_scored)

print("Validation PD decile summary:")
display(valid_decile_summary)

print("Test PD decile summary:")
display(test_decile_summary)

Validation PD decile summary:


,risk_decile,n_loans,avg_pd,actual_default_rate,defaults,lift
0,D1,5941,0.4397,0.4449,2643,2.2682
1,D2,5940,0.3121,0.3207,1905,1.6352
2,D3,5940,0.2520,0.2524,1499,1.2867
3,D4,5941,0.2121,0.2224,1321,1.1337
4,D5,5940,0.1818,0.1805,1072,0.9202
5,D6,5940,0.1571,0.1584,941,0.8077
6,D7,5941,0.1350,0.1299,772,0.6625
7,D8,5940,0.1140,0.1170,695,0.5966
8,D9,5940,0.0924,0.0825,490,0.4206
9,D10,5941,0.0628,0.0527,313,0.2686


Test PD decile summary:


,risk_decile,n_loans,avg_pd,actual_default_rate,defaults,lift
0,D1,5941,0.4404,0.4499,2673,2.2940
1,D2,5940,0.3116,0.3258,1935,1.6609
2,D3,5941,0.2519,0.2548,1514,1.2993
3,D4,5940,0.2122,0.2098,1246,1.0695
4,D5,5941,0.1826,0.1895,1126,0.9664
5,D6,5940,0.1575,0.1492,886,0.7605
6,D7,5940,0.1350,0.1401,832,0.7142
7,D8,5941,0.1139,0.1072,637,0.5467
8,D9,5940,0.0925,0.0859,510,0.4378
9,D10,5941,0.0627,0.0491,292,0.2506


# 5. Automatic PD Thresholds with Constrained Decision Tree

In [8]:
# Automatic PD binning using constrained decision tree

X_valid_pd = valid_scored[["calibrated_pd"]].copy()
y_valid = valid_scored["default_flag"].copy()

min_leaf_fraction = 0.05
min_samples_leaf = int(len(valid_scored) * min_leaf_fraction)

tree_binner = DecisionTreeClassifier(
    max_leaf_nodes=8,
    min_samples_leaf=min_samples_leaf,
    criterion="entropy",
    random_state=42
)

tree_binner.fit(X_valid_pd, y_valid)

raw_thresholds = tree_binner.tree_.threshold
thresholds = sorted([t for t in raw_thresholds if t != -2])

print("Number of thresholds learned:", len(thresholds))
print("Thresholds:")
for t in thresholds:
    print(f"{t:.6f}")

print("\nMinimum samples per leaf:", min_samples_leaf)

Number of thresholds learned: 7
Thresholds:
0.067724
0.103526
0.144593
0.194832
0.239949
0.304032
0.395704

Minimum samples per leaf: 2970


In [9]:
# Create grade lables dynamically

def get_grade_labels(n_bins):
    """
    Return business-friendly grade labels based on number of bins.
    Highest quality label is assigned to lowest PD bin.
    """
    if n_bins == 8:
        return ["AAA", "AA", "A", "BBB", "BB", "B", "CCC", "D"]
    elif n_bins == 7:
        return ["AAA", "AA", "A", "BBB", "BB", "B", "D"]
    elif n_bins == 6:
        return ["AAA", "AA", "A", "BB", "B", "D"]
    elif n_bins == 5:
        return ["AAA", "A", "BBB", "B", "D"]
    elif n_bins == 4:
        return ["Low", "Moderate", "High", "Very High"]
    else:
        return [f"Grade_{i+1}" for i in range(n_bins)]


n_bins = len(thresholds) + 1
grade_labels = get_grade_labels(n_bins)

print("Number of bins:", n_bins)
print("Grade labels:", grade_labels)

Number of bins: 8
Grade labels: ['AAA', 'AA', 'A', 'BBB', 'BB', 'B', 'CCC', 'D']


In [10]:
# Assign internal grade

def assign_internal_grade(pd_value, thresholds, grade_labels):
    """
    Assign internal grade based on calibrated PD thresholds.
    Thresholds must be sorted ascending.
    """
    if pd.isna(pd_value):
        return np.nan
    
    for i, threshold in enumerate(thresholds):
        if pd_value <= threshold:
            return grade_labels[i]
    
    return grade_labels[-1]


for df in [train_scored, valid_scored, test_scored]:
    df["internal_grade"] = df["calibrated_pd"].apply(
        lambda x: assign_internal_grade(x, thresholds, grade_labels)
    )

display(train_scored[["calibrated_pd", "internal_grade", "default_flag"]].head(20))

,calibrated_pd,internal_grade,default_flag
0,0.1010,AA,0
1,0.1436,A,0
2,0.0820,AA,0
3,0.1813,BBB,0
4,0.3559,CCC,0
5,0.0591,AAA,0
6,0.1567,BBB,0
7,0.2541,B,1
8,0.0908,AA,0
9,0.1655,BBB,0


In [11]:
# Grade summary function

def grade_summary(df, split_name, grade_col="internal_grade", pd_col="calibrated_pd"):
    summary = (
        df
        .groupby(grade_col)
        .agg(
            n_loans=("default_flag", "size"),
            default_rate=("default_flag", "mean"),
            avg_pd=(pd_col, "mean"),
            min_pd=(pd_col, "min"),
            max_pd=(pd_col, "max"),
            avg_loan_amnt=("loan_amnt", "mean"),
            avg_int_rate=("int_rate", "mean")
        )
        .reset_index()
    )
    
    # Sort according to grade label order
    grade_order = {grade: i for i, grade in enumerate(grade_labels)}
    summary["grade_order"] = summary[grade_col].map(grade_order)
    summary = summary.sort_values("grade_order").drop(columns=["grade_order"])
    
    summary.insert(0, "split", split_name)
    
    return summary


train_grade_summary = grade_summary(train_scored, "train")
valid_grade_summary = grade_summary(valid_scored, "valid")
test_grade_summary = grade_summary(test_scored, "test")

print("Train grade summary:")
display(train_grade_summary)

print("Validation grade summary:")
display(valid_grade_summary)

print("Test grade summary:")
display(test_grade_summary)

Train grade summary:


,split,internal_grade,n_loans,default_rate,avg_pd,min_pd,max_pd,avg_loan_amnt,avg_int_rate
2,train,AAA,15823,0.0398,0.0538,0.0167,0.0677,"12,179.6673",8.6466
1,train,AA,39593,0.0732,0.0870,0.0677,0.1035,"12,959.0565",10.6522
0,train,A,53997,0.1174,0.1239,0.1035,0.1446,"13,029.2076",12.1618
5,train,BBB,55544,0.1651,0.1684,0.1446,0.1948,"13,129.9137",13.3991
4,train,BB,35738,0.2223,0.2160,0.1948,0.2399,"13,834.1485",14.5356
3,train,B,32407,0.2777,0.2691,0.2399,0.3040,"15,308.2382",15.6896
6,train,CCC,25222,0.3599,0.3444,0.3040,0.3957,"17,304.3474",17.2103
7,train,D,18897,0.4929,0.4719,0.3957,0.8255,"18,319.7941",18.9649


Validation grade summary:


,split,internal_grade,n_loans,default_rate,avg_pd,min_pd,max_pd,avg_loan_amnt,avg_int_rate
2,valid,AAA,3358,0.0381,0.0539,0.0212,0.0677,"12,159.1126",8.5802
1,valid,AA,8516,0.0790,0.0869,0.0677,0.1035,"12,821.4713",10.6879
0,valid,A,11555,0.1223,0.1239,0.1035,0.1446,"13,060.8762",12.2307
5,valid,BBB,12028,0.1677,0.1684,0.1446,0.1948,"13,248.0421",13.3932
4,valid,BB,7620,0.2232,0.2161,0.1948,0.2399,"13,801.1450",14.5252
3,valid,B,6887,0.2712,0.2694,0.2400,0.3040,"15,238.6743",15.6257
6,valid,CCC,5472,0.3575,0.3442,0.3040,0.3957,"17,454.5230",17.2508
7,valid,D,3968,0.4776,0.4721,0.3957,0.7474,"18,454.6938",18.9498


Test grade summary:


,split,internal_grade,n_loans,default_rate,avg_pd,min_pd,max_pd,avg_loan_amnt,avg_int_rate
2,test,AAA,3415,0.0419,0.0541,0.0171,0.0677,"12,087.1742",8.7115
1,test,AA,8488,0.0778,0.0871,0.0677,0.1035,"12,800.7628",10.6999
0,test,A,11518,0.1235,0.1239,0.1035,0.1446,"13,079.4669",12.2348
5,test,BBB,11905,0.1661,0.1687,0.1446,0.1948,"13,281.6254",13.3798
4,test,BB,7727,0.2170,0.2158,0.1948,0.2399,"13,900.1262",14.5474
3,test,B,6987,0.2779,0.2690,0.2400,0.3040,"15,184.4533",15.7039
6,test,CCC,5333,0.3531,0.3445,0.3041,0.3957,"17,418.3058",17.2075
7,test,D,4032,0.4826,0.4718,0.3957,0.7679,"18,235.5655",19.0741


In [12]:
# Check monotonicity

def check_monotonicity(summary_df, metric="default_rate"):
    values = summary_df[metric].values
    diffs = np.diff(values)
    is_monotonic_increasing = np.all(diffs >= 0)
    
    result = {
        "is_monotonic_increasing": is_monotonic_increasing,
        "min_diff": diffs.min() if len(diffs) > 0 else np.nan,
        "diffs": diffs
    }
    
    return result


monotonicity_results = []

for split_name, summary in {
    "train": train_grade_summary,
    "valid": valid_grade_summary,
    "test": test_grade_summary
}.items():
    result = check_monotonicity(summary, metric="default_rate")
    monotonicity_results.append({
        "split": split_name,
        "is_monotonic_default_rate": result["is_monotonic_increasing"],
        "min_diff": result["min_diff"]
    })
    
    print("=" * 80)
    print(split_name.upper())
    print("Monotonic default rate:", result["is_monotonic_increasing"])
    print("Diffs:", result["diffs"])

monotonicity_df = pd.DataFrame(monotonicity_results)
display(monotonicity_df)

TRAIN
Monotonic default rate: True
Diffs: [0.0333793  0.04423769 0.04762588 0.05719809 0.05539964 0.08226781
 0.13301151]
VALID
Monotonic default rate: True
Diffs: [0.04090979 0.04325701 0.04540733 0.05553629 0.04800731 0.08622048
 0.12011442]
TEST
Monotonic default rate: True
Diffs: [0.03588275 0.04578892 0.04251892 0.05096651 0.06091357 0.07513981
 0.12955432]


,split,is_monotonic_default_rate,min_diff
0,train,True,0.0334
1,valid,True,0.0409
2,test,True,0.0359


In [13]:
# Bootstrap confidence intervals for test default rate by grade

def bootstrap_default_rate_ci(
    df,
    grade_col="internal_grade",
    target_col="default_flag",
    n_bootstrap=500,
    ci=0.95,
    random_state=42
):
    rng = np.random.default_rng(random_state)
    results = []
    
    alpha = 1 - ci
    
    for grade in grade_labels:
        grade_df = df[df[grade_col] == grade]
        
        if len(grade_df) == 0:
            continue
        
        y = grade_df[target_col].values
        boot_rates = []
        
        for _ in range(n_bootstrap):
            sample = rng.choice(y, size=len(y), replace=True)
            boot_rates.append(sample.mean())
        
        lower = np.quantile(boot_rates, alpha / 2)
        upper = np.quantile(boot_rates, 1 - alpha / 2)
        
        results.append({
            "grade": grade,
            "n_loans": len(grade_df),
            "actual_default_rate": y.mean(),
            "ci_lower": lower,
            "ci_upper": upper
        })
    
    return pd.DataFrame(results)


test_grade_ci = bootstrap_default_rate_ci(
    test_scored,
    n_bootstrap=500,
    ci=0.95,
    random_state=42
)

display(test_grade_ci)

,grade,n_loans,actual_default_rate,ci_lower,ci_upper
0,AAA,3415,0.0419,0.0359,0.0486
1,AA,8488,0.0778,0.0722,0.0837
2,A,11518,0.1235,0.1169,0.1295
3,BBB,11905,0.1661,0.1591,0.1727
4,BB,7727,0.2170,0.2090,0.2265
5,B,6987,0.2779,0.2681,0.2878
6,CCC,5333,0.3531,0.3404,0.3654
7,D,4032,0.4826,0.4689,0.4970


In [14]:
# Compare internal grade vs Lendng Club grade

if "grade" in test_scored.columns:
    internal_vs_lc = pd.crosstab(
        test_scored["internal_grade"],
        test_scored["grade"],
        normalize="index"
    ) * 100
    
    # Sort index by grade labels
    internal_vs_lc = internal_vs_lc.reindex(grade_labels)
    
    display(internal_vs_lc)
else:
    print("Lending Club grade column not available.")

grade,A,B,C,D,E,F,G
internal_grade,,,,,,,
AAA,63.8360,26.7350,7.2914,1.7277,0.3514,0.0586,0.0000
AA,36.4043,38.4189,18.0254,5.8789,1.0485,0.2003,0.0236
A,19.4739,40.6928,25.5166,11.1130,2.6480,0.4775,0.0781
BBB,11.3818,35.5901,31.4406,15.7077,4.6871,1.0500,0.1428
BB,5.7073,28.1610,34.6577,20.8619,8.0368,2.0965,0.4788
B,2.7050,20.0372,34.2636,24.2593,13.5251,4.3366,0.8730
CCC,1.0126,11.0069,29.3456,27.5455,19.9700,8.7568,2.3626
D,0.2480,3.7202,18.1300,27.1081,29.6875,15.8234,5.2827


In [15]:
# Create threshold table

threshold_rows = []

lower = 0.0

for i, grade in enumerate(grade_labels):
    if i < len(thresholds):
        upper = thresholds[i]
    else:
        upper = 1.0
    
    threshold_rows.append({
        "grade": grade,
        "pd_lower_bound": lower,
        "pd_upper_bound": upper
    })
    
    lower = upper

grade_thresholds_df = pd.DataFrame(threshold_rows)

display(grade_thresholds_df)

,grade,pd_lower_bound,pd_upper_bound
0,AAA,0.0000,0.0677
1,AA,0.0677,0.1035
2,A,0.1035,0.1446
3,BBB,0.1446,0.1948
4,BB,0.1948,0.2399
5,B,0.2399,0.3040
6,CCC,0.3040,0.3957
7,D,0.3957,1.0000


In [16]:
# Combine grade summaries

all_grade_summary = pd.concat(
    [train_grade_summary, valid_grade_summary, test_grade_summary],
    axis=0,
    ignore_index=True
)

display(all_grade_summary)

,split,internal_grade,n_loans,default_rate,avg_pd,min_pd,max_pd,avg_loan_amnt,avg_int_rate
0,train,AAA,15823,0.0398,0.0538,0.0167,0.0677,"12,179.6673",8.6466
1,train,AA,39593,0.0732,0.0870,0.0677,0.1035,"12,959.0565",10.6522
2,train,A,53997,0.1174,0.1239,0.1035,0.1446,"13,029.2076",12.1618
3,train,BBB,55544,0.1651,0.1684,0.1446,0.1948,"13,129.9137",13.3991
4,train,BB,35738,0.2223,0.2160,0.1948,0.2399,"13,834.1485",14.5356
5,train,B,32407,0.2777,0.2691,0.2399,0.3040,"15,308.2382",15.6896
6,train,CCC,25222,0.3599,0.3444,0.3040,0.3957,"17,304.3474",17.2103
7,train,D,18897,0.4929,0.4719,0.3957,0.8255,"18,319.7941",18.9649
8,valid,AAA,3358,0.0381,0.0539,0.0212,0.0677,"12,159.1126",8.5802
9,valid,AA,8516,0.0790,0.0869,0.0677,0.1035,"12,821.4713",10.6879


# 6. Save Scored Datasets with Internal Grades

In [17]:
train_grade_path = PROCESSED_DIR / "train_scored_with_grades.csv"
valid_grade_path = PROCESSED_DIR / "valid_scored_with_grades.csv"
test_grade_path = PROCESSED_DIR / "test_scored_with_grades.csv"

train_scored.to_csv(train_grade_path, index=False)
valid_scored.to_csv(valid_grade_path, index=False)
test_scored.to_csv(test_grade_path, index=False)

print("Saved scored datasets with internal grades:")
for path in [train_grade_path, valid_grade_path, test_grade_path]:
    print(f"{path} | {path.stat().st_size / (1024 ** 2):,.2f} MB")

Saved scored datasets with internal grades:
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed\train_scored_with_grades.csv | 65.15 MB
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed\valid_scored_with_grades.csv | 13.96 MB
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed\test_scored_with_grades.csv | 13.96 MB


In [18]:
grade_thresholds_path = DATA_ARTIFACT_DIR / "internal_grade_thresholds.csv"
grade_summary_path = DATA_ARTIFACT_DIR / "internal_grade_summary.csv"
test_grade_ci_path = DATA_ARTIFACT_DIR / "internal_grade_test_ci.csv"
monotonicity_path = REPORTS_DIR / "04_grade_monotonicity.csv"
internal_vs_lc_path = REPORTS_DIR / "04_internal_vs_lending_club_grade.csv"

grade_thresholds_df.to_csv(grade_thresholds_path, index=False)
all_grade_summary.to_csv(grade_summary_path, index=False)
test_grade_ci.to_csv(test_grade_ci_path, index=False)
monotonicity_df.to_csv(monotonicity_path, index=False)

if "internal_vs_lc" in globals():
    internal_vs_lc.to_csv(internal_vs_lc_path)

print("Saved artifacts:")
print(grade_thresholds_path)
print(grade_summary_path)
print(test_grade_ci_path)
print(monotonicity_path)

if "internal_vs_lc" in globals():
    print(internal_vs_lc_path)

Saved artifacts:
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\artifacts\data\internal_grade_thresholds.csv
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\artifacts\data\internal_grade_summary.csv
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\artifacts\data\internal_grade_test_ci.csv
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\artifacts\reports\04_grade_monotonicity.csv
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\artifacts\reports\04_internal_vs_lending_club_grade.csv
